# VAE Training with Weights & Biases Tracking

This notebook demonstrates how to train a VAE with **real-time experiment tracking** using [Weights & Biases](https://wandb.ai/) (wandb).

**What's new compared to `04_vae_training.ipynb`:**
- `WandbConfig` enables automatic scalar metric logging (loss, recon, KL, MMD)
- `on_epoch_end` callback logs custom visualizations:
  - Reconstruction plots (original vs reconstructed spectra)
  - Latent corner plots colored by sky condition
  - Individual broadband magnitude CDFs (V, g, r, z) — real vs reconstructed
  - Individual airglow line CDFs (OI 5577, Na I D, OH, O2, etc.) with EMD
- Metrics organized into three wandb sections: `train/`, `val/`, `viz/`
- `tqdm` progress bar with live metrics
- Checkpoint naming uses wandb run name for unique identification
- Optional: hyperparameter sweep example

**Requirements:**
```bash
pip install desisky[wandb,viz]
# or: pip install desisky[all]
```

## 1. Imports

In [1]:
import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np
import equinox as eqx
import matplotlib.pyplot as plt
import torch
from torch.utils.data import random_split
import wandb

from desisky.data import SkySpecVAC
from desisky.models.vae import make_SkyVAE
from desisky.training import (
    VAETrainer,
    VAETrainingConfig,
    NumpyLoader,
    WandbConfig,
)
from desisky.training.wandb_utils import log_figure
from desisky.visualization import (
    plot_vae_reconstructions,
    plot_latent_corner,
    plot_broadband_cdfs,
    plot_airglow_cdfs,
)

print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")

/home/mdowicz/miniconda3/envs/desisky_dev/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


JAX version: 0.7.1
JAX devices: [CudaDevice(id=0), CudaDevice(id=1)]


## 2. wandb Login

In [2]:
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/mdowicz/.netrc.
wandb: Currently logged in as: mdowicz to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 3. Load Data

In [3]:
vac = SkySpecVAC(version="v1.0", download=True)
wavelength, flux, metadata = vac.load()
flux = flux.astype(np.float32)

print(f"Loaded {flux.shape[0]:,} sky spectra")
print(f"Wavelength range: {wavelength.min():.1f} - {wavelength.max():.1f} A")

Loaded 9,176 sky spectra
Wavelength range: 3600.0 - 9824.0 A


## 4. Classify Sky Conditions

Classify each exposure as dark, moon, twilight, or other for latent space coloring.

In [4]:
def classify_sky_condition(meta):
    conditions = np.full(len(meta), "other", dtype=object)
    conditions[(meta["SUNALT"] < -20) & (meta["MOONALT"] < -5)] = "dark"
    conditions[
        (meta["SUNALT"] < -20)
        & (meta["MOONALT"] > 5)
        & (meta["MOONFRAC"] > 0.5)
        & (meta["MOONSEP"] <= 90)
    ] = "moon"
    conditions[(meta["SUNALT"] > -20) & (meta["MOONALT"] <= -5)] = "twilight"
    return conditions

sky_conditions = classify_sky_condition(metadata)
for cond in ["dark", "moon", "twilight", "other"]:
    print(f"  {cond}: {(sky_conditions == cond).sum():,}")

  dark: 4,177
  moon: 2,290
  twilight: 732
  other: 1,977


## 5. Create Train/Test Split and Data Loaders

In [5]:
dataset_size = len(flux)
train_size = int(0.9 * dataset_size)
test_size = dataset_size - train_size

flux_tensor = torch.from_numpy(flux)
generator = torch.Generator().manual_seed(32)
train_set, test_set = random_split(flux_tensor, [train_size, test_size], generator=generator)

batch_size = 64
train_loader = NumpyLoader(dataset=train_set, batch_size=batch_size, shuffle=True)
test_loader = NumpyLoader(dataset=test_set, batch_size=batch_size, shuffle=False)

# Keep test data for visualizations
test_indices = test_set.indices
test_flux = flux[test_indices]
test_conditions = sky_conditions[test_indices]

print(f"Training: {len(train_set):,} spectra ({len(train_loader)} batches)")
print(f"Test: {len(test_set):,} spectra ({len(test_loader)} batches)")

Training: 8,258 spectra (130 batches)
Test: 918 spectra (15 batches)


## 6. Initialize VAE Model

In [6]:
in_channels = flux.shape[1]
latent_dim = 8

model = make_SkyVAE(in_channels=in_channels, latent_dim=latent_dim, key=jr.PRNGKey(42))

print(f"VAE Architecture:")
print(f"  Input channels: {in_channels}")
print(f"  Latent dimension: {latent_dim}")

VAE Architecture:
  Input channels: 7781
  Latent dimension: 8


## 7. Configure Training + wandb

In [7]:
config = VAETrainingConfig(
    epochs=100,
    learning_rate=1e-4,
    beta=1e-3,
    lam=4.0,
    kernel_sigma="auto",
    save_best=True,
    run_name="sky_vae_wandb",
    validate_every=1,
    random_seed=42,
)

wconfig = WandbConfig(
    project="desisky-vae",
    run_name=None,          # Auto-generate unique name
    tags=["vae", "infovae", "full-dataset"],
    log_every=1,
    viz_every=10,
)

## 8. Define Visualization Callback

The callback logs four types of visualizations every `viz_every` epochs:
1. **Reconstruction plot** — original vs reconstructed spectra
2. **Latent corner plot** — latent dimensions colored by sky condition
3. **Broadband CDF** — CDF comparison of V, g, r, z magnitudes (real vs reconstructed) with EMD
4. **Airglow CDFs** — one CDF figure per emission line (OI 5577, Na I D, OI doublet, OH, O2(0-1), NI 5200) with EMD

In [8]:
wl_np = np.array(wavelength)


def on_epoch_end(model, history, epoch):
    if epoch % wconfig.viz_every != 0:
        return

    key = jr.PRNGKey(epoch)
    n_show = 5
    n_cdf = min(300, len(test_flux))

    # --- 1. Reconstruction plot ---
    test_batch = jnp.array(test_flux[:n_show])
    result = model(test_batch, key)
    originals = np.array(test_batch)
    recons = np.array(result["output"])

    fig_recon = plot_vae_reconstructions(originals, recons, wl_np, n_samples=n_show)
    log_figure("viz/reconstructions", fig_recon, epoch)
    plt.close(fig_recon)

    # --- 2. Latent corner plot ---
    n_enc = min(500, len(test_flux))
    enc_result = model(jnp.array(test_flux[:n_enc]), jr.PRNGKey(epoch + 1))
    latents = np.array(enc_result["latent"])

    fig_corner = plot_latent_corner(
        latents,
        labels=[f"z{i}" for i in range(latents.shape[1])],
        sky_conditions=test_conditions[:n_enc],
        condition_names=["dark", "moon", "other", "twilight"],
    )
    log_figure("viz/latent_corner", fig_corner, epoch)
    plt.close(fig_corner)

    # --- Reconstruct a larger batch for CDF comparisons ---
    cdf_batch = jnp.array(test_flux[:n_cdf])
    cdf_result = model(cdf_batch, jr.PRNGKey(epoch + 2))
    real_spectra = np.array(cdf_batch)
    recon_spectra = np.array(cdf_result["output"])

    # --- 3. Broadband CDFs (one figure per filter: V, g, r, z) ---
    bb_results = plot_broadband_cdfs(wl_np, real_spectra, recon_spectra)
    for band_name, (fig, emd) in bb_results.items():
        log_figure(f"viz/{band_name}", fig, epoch)
        plt.close(fig)

    # --- 4. Airglow CDFs (one figure per emission line) ---
    ag_results = plot_airglow_cdfs(wl_np, real_spectra, recon_spectra)
    for line_name, (fig, emd) in ag_results.items():
        safe_name = line_name.replace(" ", "_").replace("(", "-").replace(")", "-")
        log_figure(f"viz/{safe_name}", fig, epoch)
        plt.close(fig)

print(f"Callback defined. Logs reconstructions + corner"
      f" + broadband CDFs + airglow CDFs every {wconfig.viz_every} epochs.")

Callback defined. Logs reconstructions + corner + broadband CDFs + airglow CDFs every 10 epochs.


## 9. Train with wandb Tracking

In [9]:
trainer = VAETrainer(
    model, config,
    wandb_config=wconfig,
    on_epoch_end=on_epoch_end,
)

print("Starting training...")
trained_model, history = trainer.train(train_loader, test_loader)

print(f"Best test loss: {history.best_test_loss:.6f} (epoch {history.best_epoch})")
print(f"Final reconstruction MSE: {history.test_recon[-1]:.6f}")

Starting training...


  wandb run: https://wandb.ai/mdowicz/desisky-vae/runs/h002dflo


VAE Training:  10%|█         | 10/100 [00:52<02:07,  1.42s/it, best=4.3572, test=4.3572, train=5.6503]    /home/mdowicz/miniconda3/envs/desisky_dev/lib/python3.11/site-packages/speclite/filters.py:1119: RuntimeWarning: invalid value encountered in log10
  return -2.5 * np.log10(maggies)
VAE Training: 100%|██████████| 100/100 [03:06<00:00,  1.86s/it, best=0.5797, test=0.6021, train=0.5729]

  wandb run: https://wandb.ai/mdowicz/desisky-vae/runs/h002dflo


epoch,▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
train/kl,▂▁▄▅▅▆▆██▇▇▇▇▆▇███▇▇▆▆▆▅▅▅▅▅▄▄▄▄▄▄▄▄▄▄▄▃
train/loss,█▆▅▅▅▃▃▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss_z,▁▃▅▆▆▇▇▇▇█▇▇▇▇████▇▇▇▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▆▅
train/mmd,▁▆▇▆▇███▇█▇▇▆▆▇█▇▇▇▇▇▇▇▇▇▇▇▇▇▇█▇█▇██████
train/recon,█▆▅▅▅▃▃▃▃▂▂▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/kl,▁▄▄▅▅▅▆▆██▇▇▇▇▇███▇▇▆▆▆▅▅▅▅▅▅▅▅▅▄▄▄▄▄▄▄▄
val/loss,█▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/loss_z,▁▂▂▂▁▃▄▄█▆▅▅▅▅▅▅▄▇▆▅▅▄▃▃▃▃▃▃▂▂▂▂▂▁▂▁▂▁▁▁
val/mmd,▁▂▃▃▃▄▄▆▅▄▅▄▄▅▄▄▃▄▅▄▆▅▃▆▆▅▅▅▄▄▆▆▄▄▆▆▅▇█▇
+1,...


Best test loss: 0.579697 (epoch 98)
Final reconstruction MSE: 0.458169


## 10. Summary

All metrics and plots are available on your wandb dashboard, organized into three sections:

**`train/` — Training scalars (every epoch):**
- `train/loss` — total InfoVAE loss
- `train/recon` — reconstruction MSE
- `train/kl` — weighted KL divergence
- `train/mmd` — weighted MMD
- `train/loss_z` — combined latent regularization (KL + MMD)

**`val/` — Validation scalars (every epoch):**
- `val/loss`, `val/recon`, `val/kl`, `val/mmd`, `val/loss_z`

**`viz/` — Figures (every `viz_every` epochs):**
- `viz/reconstructions` — original vs reconstructed spectra
- `viz/latent_corner` — latent space colored by sky condition
- `viz/V`, `viz/g`, `viz/r`, `viz/z` — individual broadband magnitude CDFs with EMD
- `viz/OI_5577`, `viz/Na_I_D`, `viz/OI_doublet`, `viz/OH`, `viz/O2-0-1-`, `viz/NI_5200` — individual airglow line CDFs with EMD

## 11. Optional: Hyperparameter Sweep

In [10]:
sweep_config = {
    "method": "grid",
    "metric": {"name": "val/loss", "goal": "minimize"},
    "parameters": {
        "beta": {"values": [1e-4, 1e-3]},
        "lam": {"values": [2.0, 4.0]},
    },
}

def train_sweep(config=None):
    # Must init before accessing wandb.config (sweep agent populates it)
    wandb.init()

    sweep_train_config = VAETrainingConfig(
        epochs=50,
        learning_rate=1e-4,
        beta=wandb.config.beta,
        lam=wandb.config.lam,
        save_best=False,
        validate_every=1,
        random_seed=42,
    )

    sweep_wconfig = WandbConfig(
        project="desisky-vae",
        tags=["sweep", "beta-lam"],
        log_every=1,
        viz_every=25,
    )

    sweep_model = make_SkyVAE(
        in_channels=in_channels, latent_dim=latent_dim, key=jr.PRNGKey(42),
    )
    sweep_trainer = VAETrainer(
        sweep_model, sweep_train_config,
        wandb_config=sweep_wconfig,
        on_epoch_end=on_epoch_end,
    )
    sweep_trainer.train(train_loader, test_loader)

# Uncomment to run:
sweep_id = wandb.sweep(sweep_config, project="desisky-vae")
wandb.agent(sweep_id, train_sweep, count=9)

Create sweep with ID: kynkwelo
Sweep URL: https://wandb.ai/mdowicz/desisky-vae/sweeps/kynkwelo


wandb: Agent Starting Run: 67whe7s5 with config:
wandb: 	beta: 0.0001
wandb: 	lam: 2
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/mdowicz/.netrc.


  wandb run: https://wandb.ai/mdowicz/desisky-vae/runs/67whe7s5


VAE Training: 100%|██████████| 50/50 [01:21<00:00,  1.62s/it, best=3.8490, test=3.8490, train=3.7091]         

  wandb run: https://wandb.ai/mdowicz/desisky-vae/runs/67whe7s5


...

  wandb run: https://wandb.ai/mdowicz/desisky-vae/runs/hiwkqv0q


VAE Training: 100%|██████████| 50/50 [01:18<00:00,  1.56s/it, best=3.5985, test=3.5985, train=3.3650]         

  wandb run: https://wandb.ai/mdowicz/desisky-vae/runs/hiwkqv0q


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train/kl,▂▁▁▁▁▁▂▅▅▅▆▆▇▇██████▇▇▇▇▇▆▆▆▆▆▅▅▅▅▅▅▅▅▅▅
train/loss,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss_z,▃▄▃▁▃▃▅▇▇▇▇▇▇████████████▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
train/mmd,▃▅▄▁▃▅▇█████████████████████████████████
train/recon,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/kl,▁▁▁▁▁▁▄▅▅▅▆▇▇▇███▇█▇▇▇▆▆▆▆▆▆▆▅▆▅▅▅▅▅▅▄▅▅
val/loss,█▇▇▇▇▄▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val/loss_z,▃▃▁▁▃▆▇▇▇▇▇███████████▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
val/mmd,▅▅▁▁▂▇██████████████████████████████████
+1,...


wandb: Sweep Agent: Waiting for job.
wandb: Sweep Agent: Exiting.
